# No-graph pixel-time comparison: folds 0 and 3

Train **no graph only**, on **folds 0 and 3**, and compare with Marcin's recorded learned-graph results on those same folds. This is a two-fold comparison, not the full 12-fold benchmark. No graph removes neighbour messages, not the spatial pixel CNN. The architecture and training recipe otherwise follow the learned model; removing attention reduces the parameter count.

Attach your private cache dataset, enable a GPU and Internet, and add an enabled Kaggle Secret named `GITHUB_TOKEN` (read-only access to the private repository). Never paste the token into this notebook.

**The ablation branch must be committed and pushed to GitHub before the clone cell will work.** Run cells in order. Full training is opt-in below. The smoke test is not an accuracy experiment.

In [ ]:
from pathlib import Path

BRANCH = 'wishaal/pixel-time-ablations-kaggle'
REPO_DIR = Path('/kaggle/working/GraphMLWildfire-logged')  # Fresh checkout with live logging.
DATA_DIR = Path('/kaggle/input/datasets/wkanhai/zip-graphmlwildfire-train-cache/training_cache')
OUTPUT_DIR = Path('/kaggle/working/pixel_time_ablations')

FOLDS_TO_RUN = [0, 3]  # Compare only these same folds against Marcin's learned model.
SEEDS = [0]
MODES = ['none']
EPOCHS = 30  # Keep this equal across all variants.
BATCH_SIZE = 32  # Match Marcin's training batch directly; may exceed T4 memory.
EVAL_BATCH_SIZE = 1
PRECISION = 'auto'  # Native BF16 when supported; FP16 + gradient scaling on Tesla T4.
RUN_TRAINING = False  # Set True only after the smoke test succeeds.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Clone the code
Authentication is provided only through the child process environment, not the URL, notebook output, or saved Git config. An existing checkout must be clean and on the requested branch; this cell never overwrites or automatically updates it. To use newer code, use a fresh directory or deliberately update your checkout.

In [ ]:
import base64
import os
import subprocess
import sys

def git_output(*args):
    return subprocess.check_output(['git', '-C', str(REPO_DIR), *args], text=True).strip()

def clone_code():
    if REPO_DIR.exists():
        if git_output('branch', '--show-current') != BRANCH:
            raise RuntimeError('Existing checkout is on a different branch; use a new REPO_DIR.')
        if git_output('status', '--porcelain', '--untracked-files=no'):
            raise RuntimeError('Existing checkout has tracked edits; inspect them before proceeding.')
        print('Using existing checkout; not fetching updates.')
        return
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    env = os.environ.copy()
    env.update({
        'GIT_TERMINAL_PROMPT': '0',
        'GIT_CONFIG_COUNT': '2',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'Authorization: Basic {auth}',
        'GIT_CONFIG_KEY_1': 'credential.helper',
        'GIT_CONFIG_VALUE_1': '',
    })
    subprocess.run([
        'git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH,
        'https://github.com/berkentekin/GraphMLWildfire.git', str(REPO_DIR),
    ], env=env, check=True)

clone_code()
CODE_COMMIT = git_output('rev-parse', 'HEAD')
print('Branch:', BRANCH)
print('Commit:', CODE_COMMIT)


## 2. Use Kaggle's existing GPU PyTorch
Do not run `uv sync` or upgrade PyTorch here. Install only missing runtime dependencies, then install this repository without resolving its training-machine version pins. Import errors other than missing top-level packages are surfaced rather than hidden.

In [ ]:
import importlib
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Kaggle notebook settings first.')
packages = ['numpy', 'h5py', 'rasterio', 'affine', 'tqdm', 'wandb', 'pandas']
missing = []
for name in packages:
    try:
        importlib.import_module(name)
    except ModuleNotFoundError as exc:
        if exc.name != name:
            raise
        missing.append(name)
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', str(REPO_DIR),
], check=True)
sys.path.insert(0, str(REPO_DIR / 'src'))
os.environ['WANDB_MODE'] = 'disabled'  # Local results only; no W&B login or upload.
os.environ['PYTHONUNBUFFERED'] = '1'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
from graphml_wildfire.train import resolve_precision
print('Training precision:', resolve_precision(PRECISION, torch.device('cuda')))


In [ ]:
import json
from graphml_wildfire.benchmark import YEARS
from graphml_wildfire.data.mmap import TARGETS
from graphml_wildfire.data.pixels import PIXEL_CHANNELS

fires = sorted(DATA_DIR.glob('*/*/features.npy'))
if not fires:
    raise FileNotFoundError(f'No cache found under {DATA_DIR}; check the attached dataset path.')
required = [f'{name}.npy' for name in TARGETS] + [f'pix_{name}.npy' for name in PIXEL_CHANNELS] + ['pix_centre.npy']
missing = [str(fire.parent / name) for fire in fires for name in required if not (fire.parent / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing cache files: {missing[:10]}')
for year in YEARS:
    count = sum(fire.parent.parent.name == str(year) for fire in fires)
    if not count:
        raise ValueError(f'Missing year {year}')
    print(year, 'fires:', count)
print('Total fires:', len(fires))
environment = {
    'branch': BRANCH, 'commit': CODE_COMMIT, 'data_dir': str(DATA_DIR),
    'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
    'python': sys.version, 'precision': PRECISION,
}
environment_file = OUTPUT_DIR / 'environment.json'
if environment_file.exists() and json.loads(environment_file.read_text()) != environment:
    raise ValueError('This OUTPUT_DIR contains a different code/environment setup; use a new OUTPUT_DIR.')
environment_file.write_text(json.dumps(environment, indent=2))


## 3. Check the reference recipe, then smoke-test no graph
The reference check matches the architectural/training hyperparameters to the repository's recorded learned-graph runs. The smoke test uses the full default architecture and the configured **32-sample batch**, then one whole-fire validation sample. This does not train a useful model or produce benchmark AP. Initial cache statistics can take time.

If the test fails, stop. For FP16 non-finite gradients, try `PRECISION = 'fp32'` and rerun. If batch 32 runs out of memory, do not silently lower it and call the recipe matched: use a larger GPU or add gradient accumulation separately. A passing smoke test does not guarantee every larger validation fire fits or both folds finish within Kaggle's runtime limits.

Runtime differences remain: T4 uses FP16 rather than the original training code's BF16, evaluation batch size is 1 rather than 8, and compilation is disabled. This is a matched training recipe, not an identical hardware/numerics reproduction.

In [ ]:
from datetime import datetime, timezone
import shutil

def run_logged(command, log_file):
    log_file = Path(log_file)
    log_file.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    print(f'Log file: {log_file}', flush=True)
    with log_file.open('a', encoding='utf-8', buffering=1) as log:
        with subprocess.Popen(command, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    message = f'[{datetime.now(timezone.utc).isoformat(timespec="seconds")}] {line.rstrip()}'
                    print(message, flush=True)
                    log.write(message + '\n')
                code = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if code:
        raise subprocess.CalledProcessError(code, command)

def snapshot_results():
    archive = shutil.make_archive('/kaggle/working/pixel_time_ablations_results', 'zip', OUTPUT_DIR)
    print(f'Updated results archive: {archive}', flush=True)

print('Live logging enabled: subprocess messages are streamed into notebook cells and saved under OUTPUT_DIR/logs.', flush=True)


In [ ]:
from graphml_wildfire.train import TrainConfig

REFERENCE_FILE = REPO_DIR / 'results' / 'benchmark_runs.jsonl'
reference_records = [json.loads(line) for line in REFERENCE_FILE.read_text().splitlines() if line.strip()]
reference_records = [r for r in reference_records if r.get('model') == 'pixel_gtcnn'
                     and r.get('experiment_name') == 'pixel_gtcnn_w5_p2'
                     and r.get('graph_mode', 'learned') == 'learned']
REFERENCE_RUNS = {}
for record in reference_records:
    key = (record['fold'], record['seed'])
    if key in REFERENCE_RUNS:
        raise ValueError(f'Multiple learned references for fold/seed {key}')
    REFERENCE_RUNS[key] = record
RECIPE_FIELDS = ['model', 'window', 'hidden', 'layers', 'taps', 'heads', 'pixel_hidden', 'decoder_hidden',
                 'dropout', 'lr', 'weight_decay', 'epochs', 'batch_size', 'crop', 'val_stride', 'patch']
for fold in FOLDS_TO_RUN:
    for seed in SEEDS:
        key = (fold, seed)
        if key not in REFERENCE_RUNS:
            raise ValueError(f'No recorded learned reference for fold/seed {key}')
        planned = TrainConfig(graph_mode='none', fold=fold, seed=seed, epochs=EPOCHS, batch_size=BATCH_SIZE)
        mismatches = [field for field in RECIPE_FIELDS if getattr(planned, field) != REFERENCE_RUNS[key][field]]
        if mismatches:
            raise ValueError(f'Training recipe differs from learned reference for {key}: {mismatches}')
(OUTPUT_DIR / 'learned_reference_runs.json').write_text(json.dumps(reference_records, indent=2))
print('Training recipe matches the recorded learned-model reference for all requested folds/seeds.')
print('Runtime differences: GPU/precision, evaluation batch size, and compilation; see notes above.')


In [ ]:
SMOKE_FILE = OUTPUT_DIR / 'smoke.json'
SMOKE_PASSED = False
run_logged([
    sys.executable, '-m', 'graphml_wildfire.smoke',
    '--data-dir', str(DATA_DIR), '--precision', PRECISION,
    '--fold', str(FOLDS_TO_RUN[0]), '--batch-size', str(BATCH_SIZE),
    '--output', str(SMOKE_FILE),
    '--graph-modes', *MODES,
], OUTPUT_DIR / 'logs' / 'smoke.log')
SMOKE_PASSED = True
SMOKE_SETTINGS = (str(DATA_DIR), PRECISION, FOLDS_TO_RUN[0], BATCH_SIZE, tuple(MODES))
print('Smoke test passed. You can now opt into training in the settings cell.')


## 4. Train and evaluate no graph on folds 0 and 3
Set `RUN_TRAINING = True` in the settings cell and rerun that cell, then this one. Defaults run **two no-graph models: folds 0 and 3, 30 epochs each, seed 0, batch 32**, sequentially on one GPU. Neither uniform nor learned graph is trained. Even two folds may exceed one Kaggle session's runtime allowance; run `[0]` and `[3]` separately if necessary and preserve outputs between sessions.

The training batch size, epochs, seed, fold definitions, feature inputs, crop, optimizer, and architecture settings match the recorded learned-graph recipe. Runtime precision, evaluation batching, and compilation differ as documented above. The output comparison uses the same fold/seed reference and benchmark AP metric.

Progress is explicitly streamed into this cell and timestamped in per-run files under `OUTPUT_DIR/logs`. Training reports the first/last batch and about once per minute between batches; evaluation reports progress too. A single slow operation can still delay messages. Kaggle may delay displaying saved-version logs; interactive cell output is the most direct way to watch progress.

Best validation weights/config are saved whenever validation AP improves. These are **not full optimizer/scheduler resume checkpoints**. A results ZIP is refreshed after each fold and on a normal Python error/interrupt, not only at the end. A hard session termination can bypass this cleanup. Files in `/kaggle/working` are not external backups: download the ZIP or preserve successful version outputs. The cell skips exact runs already recorded as completed; unfinished runs start from scratch.

In [ ]:
from dataclasses import asdict
from graphml_wildfire.train import TrainConfig

RESULTS_FILE = OUTPUT_DIR / 'benchmark_runs.jsonl'
if not RUN_TRAINING:
    print('Training is disabled. Set RUN_TRAINING = True after the smoke test passes.')
else:
    if not globals().get('SMOKE_PASSED', False):
        raise RuntimeError('Run the smoke-test cell successfully first.')
    if SMOKE_SETTINGS != (str(DATA_DIR), PRECISION, FOLDS_TO_RUN[0], BATCH_SIZE, tuple(MODES)):
        raise RuntimeError('Cache/precision/initial fold/batch/modes changed; rerun the smoke test first.')
    completed = {json.loads(line)['run_name'] for line in RESULTS_FILE.read_text().splitlines()} if RESULTS_FILE.exists() else set()
    for fold in FOLDS_TO_RUN:
        for seed in SEEDS:
            for mode in MODES:
                config = TrainConfig(
                    experiment_name=f'pixel_time_{mode}', graph_mode=mode,
                    fold=fold, seed=seed, epochs=EPOCHS,
                    batch_size=BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE,
                    precision=PRECISION, compile=0, data_dir=str(DATA_DIR),
                    checkpoint_dir=str(OUTPUT_DIR / 'checkpoints'), results_file=str(RESULTS_FILE),
                )
                if config.run_name in completed:
                    print('Already complete:', config.run_name)
                    continue
                command = [sys.executable, '-m', 'graphml_wildfire.train']
                for key, value in asdict(config).items():
                    command.extend(['--' + key.replace('_', '-'), str(value)])
                print('Starting:', config.run_name, flush=True)
                try:
                    run_logged(command, OUTPUT_DIR / 'logs' / f'{config.run_name}.log')
                finally:
                    snapshot_results()
                completed.add(config.run_name)


## 5. No-graph results versus recorded learned-graph results
The main leaderboard metric is **`test_ap_centre128`**. `test_ap` scores the full benchmark crop. F1/IoU below use the existing 0.5 threshold. AP differences are absolute (0.01 = one percentage point).

Each finished no-graph fold is paired with the same fold/seed in Marcin's recorded learned-graph results. The AP difference is learned minus no graph. Both comparison means use only those matched folds, so partial runs are not compared against the reference's full 12-fold mean. A full benchmark requires all 12 folds; sample standard deviation is unavailable for a single fold. Do not claim improvement until the measured results support it.

In [ ]:
import pandas as pd
from IPython.display import display

if not RESULTS_FILE.exists():
    print('No benchmark results yet. Smoke-test losses are not benchmark results.')
else:
    rows = pd.read_json(RESULTS_FILE, lines=True).drop_duplicates('run_name', keep='last')
    rows = rows[rows.graph_mode.isin(MODES) & rows.fold.isin(FOLDS_TO_RUN) & rows.seed.isin(SEEDS)].copy()
    if rows.empty:
        raise ValueError('No completed no-graph runs match the requested folds/seeds.')
    allowed = {'graph_mode', 'experiment_name', 'fold', 'seed'}
    for field in TrainConfig.__dataclass_fields__:
        if field not in allowed and rows[field].nunique() > 1:
            raise ValueError(f'Mixed {field} settings in results; use a separate OUTPUT_DIR per experiment.')
    if rows.duplicated(['graph_mode', 'fold', 'seed']).any():
        raise ValueError('Multiple runs for one mode/fold/seed; separate experiments before comparing.')
    columns = ['graph_mode', 'fold', 'seed', 'best_epoch', 'test_ap_centre128', 'test_ap', 'test_f1', 'test_iou', 'parameters', 'minutes', 'resolved_precision']
    table = rows[columns].sort_values(['fold', 'seed', 'graph_mode'])
    display(table)
    table.to_csv(OUTPUT_DIR / 'per_fold_results.csv', index=False)
    summary = rows.groupby(['graph_mode', 'seed']).agg(
        folds=('fold', 'nunique'), mean_ap=('test_ap_centre128', 'mean'), sample_std_ap=('test_ap_centre128', 'std'),
    ).reset_index()
    display(summary)
    summary.to_csv(OUTPUT_DIR / 'summary.csv', index=False, na_rep='not available')
    for record in rows.to_dict('records'):
        key = (record['fold'], record['seed'])
        if key not in REFERENCE_RUNS:
            raise ValueError(f'Missing learned reference for {key}; rerun the reference-check cell.')
        mismatches = [field for field in RECIPE_FIELDS if record[field] != REFERENCE_RUNS[key][field]]
        if mismatches:
            raise ValueError(f'Completed run differs from reference recipe for {key}: {mismatches}')
    reference = pd.DataFrame(list(REFERENCE_RUNS.values()))[['fold', 'seed', 'test_ap_centre128']]
    paired = table[['fold', 'seed', 'test_ap_centre128']].rename(columns={'test_ap_centre128': 'no_graph_ap'}).merge(
        reference.rename(columns={'test_ap_centre128': 'learned_ap'}), on=['fold', 'seed'], validate='one_to_one',
    )
    paired['learned_minus_none'] = paired['learned_ap'] - paired['no_graph_ap']
    display(paired)
    paired.to_csv(OUTPUT_DIR / 'paired_differences.csv', index=False)
    matched_summary = paired.groupby('seed').agg(
        folds=('fold', 'nunique'), no_graph_mean_ap=('no_graph_ap', 'mean'), no_graph_std_ap=('no_graph_ap', 'std'),
        learned_mean_ap=('learned_ap', 'mean'), learned_std_ap=('learned_ap', 'std'),
        mean_learned_minus_none=('learned_minus_none', 'mean'),
    ).reset_index()
    display(matched_summary)
    matched_summary.to_csv(OUTPUT_DIR / 'matched_comparison_summary.csv', index=False, na_rep='not available')
    print(f'Completed fold/seed runs: {len(paired)}/{len(FOLDS_TO_RUN) * len(SEEDS)} requested.')
    if any(set(paired.loc[paired.seed == seed, 'fold']) != set(range(12)) for seed in SEEDS):
        print('PARTIAL benchmark: the matched comparison above does not yet cover all 12 folds per seed.')


## 6. Preserve your results
Run this after the runs you want to keep. Download the archive or save a notebook version with outputs before the session ends. Keep the notebook private while it contains private-project code/output. The ZIP contains results, provenance, smoke diagnostics, and best weights/configs, not the training cache or GitHub secret.

In [ ]:
import shutil
from IPython.display import FileLink

archive = shutil.make_archive('/kaggle/working/pixel_time_ablations_results', 'zip', OUTPUT_DIR)
display(FileLink(archive))
